In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import math
import sklearn.linear_model
import sklearn.metrics
import sklearn.neighbors
import sklearn.preprocessing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

In [6]:
df = pd.read_csv('car_data.csv')



In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   DateCrawled        354369 non-null  str  
 1   Price              354369 non-null  int64
 2   VehicleType        316879 non-null  str  
 3   RegistrationYear   354369 non-null  int64
 4   Gearbox            334536 non-null  str  
 5   Power              354369 non-null  int64
 6   Model              334664 non-null  str  
 7   Mileage            354369 non-null  int64
 8   RegistrationMonth  354369 non-null  int64
 9   FuelType           321474 non-null  str  
 10  Brand              354369 non-null  str  
 11  NotRepaired        283215 non-null  str  
 12  DateCreated        354369 non-null  str  
 13  NumberOfPictures   354369 non-null  int64
 14  PostalCode         354369 non-null  int64
 15  LastSeen           354369 non-null  str  
dtypes: int64(7), str(9)
memory usage: 43.3 MB


In [8]:
df.head()

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


In [9]:
df.describe()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000,354369.0,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645,0.0,50508.689087
std,4514.158514,90.227958,189.850405,37905.341530,3.726421,0.0,25783.096248
min,0.000000,1000.000000,0.000000,5000.000000,0.000000,0.0,1067.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000,0.0,30165.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000,0.0,49413.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000,0.0,71083.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000,0.0,99998.000000


In [10]:
# Se aplican filtros para eleliminar precios que esten menos de 1050 euros, ya que segun la estadistica nos indica que tenemos hasta precios de 0 euros
minimo = 1050
maximo = 20000
df = df[(df['Price'] >= minimo) & (df['Price'] <= maximo)]


In [11]:
# Se aplican filtros para RegistrationYear y quitar los datos fuera de tiempo
minimo = 1980
maximo = 2016
df = df[(df['RegistrationYear'] >= minimo) & (df['RegistrationYear'] <= maximo)]

In [12]:
# Se aplican filtros para Power y quitar los datos fuera de tiempo
minimo = 55
maximo = 612
df = df[(df['Power'] >= minimo) & (df['Power'] <= maximo)]

In [13]:
# Rellenar los valores Nan con 'unknown' a las columnas tipo object
type(df)

cols_object = df.select_dtypes('object')

df[cols_object.columns] =cols_object.fillna(value="unknown")

C:\Users\SUSAN RENEE GIRON\AppData\Local\Temp\ipykernel_7272\2774328845.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_object = df.select_dtypes('object')


In [14]:
#Eliminar las columnas que no suman al analisis y que solo son datos de la pagina 

df = df.drop(['DateCrawled','DateCreated','LastSeen','NumberOfPictures'],axis=1)

df.head()

,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,PostalCode
1,18300,coupe,2011,manual,190,unknown,125000,5,gasoline,audi,yes,66954
2,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,unknown,90480
3,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,91074
4,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,60437
6,2200,convertible,2004,manual,109,2_reihe,150000,8,petrol,peugeot,no,67112


In [15]:
#  Aplicar OHE a las columnas tipo object
columns_object = df.select_dtypes('object').columns
df_ohe = pd.get_dummies(df, columns = columns_object , drop_first=True)

df_ohe.info()


C:\Users\SUSAN RENEE GIRON\AppData\Local\Temp\ipykernel_7272\3487440879.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columns_object = df.select_dtypes('object').columns


<class 'pandas.DataFrame'>
Index: 229321 entries, 1 to 354368
Columns: 311 entries, Price to NotRepaired_yes
dtypes: bool(305), int64(6)
memory usage: 78.9 MB


Entrenamiento del modelo

In [16]:
# Definir features y target

target = df_ohe['Price']
features = df_ohe.drop('Price', axis=1)


# Dividir en conjuntos de entrenamiento y prueba
features_train, features_valid, target_train, target_valid = train_test_split(features,target, test_size=0.30, random_state=12345)


In [17]:
%%time
# Entrenar  modelo de regresion linear
model_LR = LinearRegression()
model_LR.fit(features_train, target_train)


CPU times: total: 3min
Wall time: 22.6 s


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](310,)","[ 376.76, 31.29, -0.04,..., 492.42, -494.05,-1865.06]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](310,)","['RegistrationYear','Power','Mileage',...,'Brand_volvo', 'NotRepaired_unknown','NotRepaired_yes']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-7.488e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,310
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(144)


In [18]:
%%time
# validar el modelo  de Linear REgression
predictions = model_LR.predict(features_valid)

CPU times: total: 594 ms
Wall time: 492 ms


In [19]:
# Evaluar modelo de regresion linear

mse_LR = mean_squared_error(target_valid, predictions)
rmse_LR = mse_LR**.5

print(f'RMSE_LR:{rmse_LR:.2f}')


RMSE_LR:2508.23


In [41]:
# Calcular los mejores parametros  de RandomForestRegressor


best_model = None
best_result = float('inf')
best_depth = 0


for depth in range(1,5): 
    model_RF =RandomForestRegressor(max_depth=depth,random_state=12345)

    model_RF.fit(features_train, target_train)



    predictions_valid = model_RF.predict(features_valid)


    result = mean_squared_error(target_valid, predictions_valid)**0.5
    # entrena el modelo en el conjunto de entrenamiento

  

    if result < best_result:
        best_model = model_RF
        best_result = result
        best_depth = depth

print(f"RMSE_RF del mejor modelo en el conjunto de validación (max_depth = {best_depth}): {best_result}")


RMSE_RF del mejor modelo en el conjunto de validación (max_depth = 4): 2586.670877865996


In [22]:
%%time
# Entrenar modelo Random Forest para medir tiempo
model_RF =RandomForestRegressor(max_depth=16,random_state=12345)
model_RF.fit(features_train, target_train)

CPU times: total: 5min 45s
Wall time: 6min 6s


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",12345
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease 

In [23]:
%%time
# validar modelo Random Forest regresor para medir tiempo
predictions_valid = model_RF.predict(features_valid)

CPU times: total: 2.19 s
Wall time: 2.45 s


In [24]:
#Evaluar modelo de random Forest Regresor
RMSE_RF = mean_squared_error(target_valid, predictions_valid)**0.5

print(f"RMSE_RF: {RMSE_RF:.2f}" )

RMSE_RF: 1672.28


In [27]:
# Elegir los mejores parametros para  LightGBM
params_list = [
    {'n_estimators': 10, 'learning_rate': 0.1, 'max_depth': 6},
    {'n_estimators': 20, 'learning_rate': 0.05, 'max_depth': 8}]


for params in params_list: 
    model_LGBM =LGBMRegressor(n_estimators=params['n_estimators'],learning_rate= params['learning_rate'],max_depth=params['max_depth'])


    model_LGBM.fit(features_train, target_train)


    predictions_valid = model_LGBM.predict(features_valid)


    result = mean_squared_error(target_valid, predictions_valid)**0.5
    # entrena el modelo en el conjunto de entrenamiento

  
   
if result < best_result:
        best_model = model_LGBM
        best_result = result
        n_estimators = params['n_estimators']
        learning_rate = params['learning_rate']
        max_depth = params['max_depth']
        print(f"RMSE_LGBM del mejor modelo en el conjunto de validación (n_estimators={n_estimators},learning_rate={learning_rate},max_depth={max_depth}): {best_result}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013786 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1102
[LightGBM] [Info] Number of data points in the train set: 160524, number of used features: 283
[LightGBM] [Info] Start training from score 5970.431892
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011706 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1102
[LightGBM] [Info] Number of data points in the train set: 160524, number of used features: 283
[LightGBM] [Info] Start training from score 5970.431892
RMSE_LGBM del mejor modelo en el conjunto de validación (n_estimators=20,learning_rate=0.05,max_depth=8): 2691.8524256332394


In [28]:
%%time
# Entrenar el modelo LGBMR con los mejores parametros

model_LGBM =LGBMRegressor(n_estimators=200,learning_rate=0.05,max_depth=8)
model_LGBM.fit(features_train, target_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006501 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1102
[LightGBM] [Info] Number of data points in the train set: 160524, number of used features: 283
[LightGBM] [Info] Start training from score 5970.431892
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
CPU times: total: 10.2 s
Wall time: 1.61 s


,max_depth,8
,learning_rate,0.05
,n_estimators,200
,boosting_type,'gbdt'
,num_leaves,31
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [29]:
%%time
#validar el modelo LGBMR con los mejores parametros

predictions_valid = model_LGBM.predict(features_valid)

CPU times: total: 2.09 s
Wall time: 390 ms


In [30]:
# evaluar el modelo LGBMR con los mejores parametros
RMSE_LGBM = mean_squared_error(target_valid, predictions_valid)**0.5

print(f'RMSE_LGBM: {RMSE_LGBM:.2f}' )

RMSE_LGBM: 1728.46


In [32]:
#  Encontrar los mejores parametros para CatBoost


best_model = None
best_result = float('inf')
best_depth = 0

for depth in range(1,10):
    model_cat = CatBoostRegressor(depth=depth, iterations=10, random_seed=12345,verbose=0)
    model_cat.fit(features_train, target_train)

    predictions_valid = model_cat.predict(features_valid)
    result = mean_squared_error(target_valid, predictions_valid)**.5
   
if result < best_result:
    best_model = model_cat
    best_result = result
    best_depth = depth
    print(f"RMSE_RF del mejor modelo en el conjunto de validación (max_depth = {best_depth}): {best_result}")


RMSE_RF del mejor modelo en el conjunto de validación (max_depth = 9): 1875.483300726846


In [33]:
%%time
#entrenar modelo CatBoost
model_cat = CatBoostRegressor(depth=9, iterations=10, random_seed=12345,verbose=0)
model_cat.fit(features_train, target_train)


CPU times: total: 4.62 s
Wall time: 754 ms


CatBoostRegressor(depth=9, iterations=10, loss_function='RMSE', random_seed=12345, verbose=0)

In [34]:
%%time
#Evaluar modelo CatBoost
predictions_valid = model_cat.predict(features_valid)

CPU times: total: 46.9 ms
Wall time: 80.3 ms


In [35]:
#Calcular RMSE de CatBoost
RMSE_cat = mean_squared_error(target_valid, predictions_valid)**.5
print(f'RMSE_cat:{RMSE_cat:.2F}')

RMSE_cat:1875.48


In [36]:
#Encontrar los mejores parametors para XGBoost


best_model = None
best_result = float('inf')
best_depth = 0

for depth in range(1,10):
    model_XGB= XGBRegressor(max_depth=depth, n_estimators=10, random_state=12345)
    model_XGB.fit(features_train, target_train)

    predictions_valid = model_XGB.predict(features_valid)
    result = mean_squared_error(target_valid, predictions_valid)**.5
   
if result < best_result:
    best_model = model_XGB
    best_result = result
    best_depth = depth
    print(f"RMSE_XGB del mejor modelo en el conjunto de validación (max_depth = {best_depth}): {best_result}")


RMSE_XGB del mejor modelo en el conjunto de validación (max_depth = 9): 1746.265157414532


In [37]:
%%time
#  Entrenar modelo  XGBoost
model_XGB= XGBRegressor(max_depth=9, n_estimators=10, random_state=12345)
model_XGB.fit(features_train, target_train)


CPU times: total: 11.6 s
Wall time: 1.6 s


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [38]:
%%time
# Evaluar modelo XGBoost 
predictions_valid = model_XGB.predict(features_valid)

CPU times: total: 500 ms
Wall time: 225 ms


In [39]:
#Calcular RMSE de XGBoost
RMSE_XGB = mean_squared_error(target_valid, predictions_valid)**.5
print(f'RMSE_XGB:  {RMSE_XGB:.2f}')


RMSE_XGB:  1746.27


Análisis del modelo¶

In [40]:
#Presentacion de resultados:

resultados = {
    'Modelo': ['LinearRegression', 'Random Forest','LightGBM','CatBoost','XGBoost'],
    'RMSE': [rmse_LR, RMSE_RF, RMSE_LGBM, RMSE_cat,RMSE_XGB],
    'Tiempo entrenamiento (s)': [5.11, 165, 3.67, .96,22],
    'Tiempo predicción (ms)': [146, 1120,513, 8.12, 327] 
}

df_resultados = pd.DataFrame(resultados).round(1)

print(df_resultados)

             Modelo    RMSE  Tiempo entrenamiento (s)  Tiempo predicción (ms)
0  LinearRegression  2508.2                       5.1                   146.0
1     Random Forest  1672.3                     165.0                  1120.0
2          LightGBM  1728.5                       3.7                   513.0
3          CatBoost  1875.5                       1.0                     8.1
4           XGBoost  1746.3                      22.0                   327.0


Conclusiones

Se selecciona el modelo CatBoost por tener buena calidad (RMSE bajo), velocidad de entrenamiento de 1 segundo y velocidad de predicion de 8 milisegundos, lo que cumple con lo solicitado por la empresa Rusty Bargain.

XGBoost tiene un RMSE de 1764.5 vs CatBoost con 1875.5, una diferencia de ~110 puntos, pero CatBoost es mas rapido:

Entrenamiento: CatBoost (1.0 s) vs XGBoost (22.0 s) → 22 veces más rápido Predicción: CatBoost (8.1 ms) vs XGBoost (327 ms) → 40 veces más rápido.

El modelo Random Forest no se selecciona por que es el mas lento, inclusive pudiera tener mayor calidad pero tambien aumentaria los tiempos de entrenamiento y prediccion. El modelo LinearRegression tiene menor calidad (RMSE mas alto). Por ultimo el Modelo LightGBM tiene una calidad media de RMSE y su tiempo tambien es regular.